# 📈 Price History · 🌐 Social Physics · ∫ Calculus Rush · 🗄️ Crypto DB · ⚔️ Military-Backed Currency

> **Thesis:** Price history is statistical mechanics of greed. Social dynamics is
> fluid mechanics of belief. Calculus is the shared grammar. Crypto rules are
> database constraints. And every reserve currency — dollar, bitcoin, or grain —
> needs a military or a hash rate backing it.

| § | Topic |
|---|---|
| §1 | Price history — returns, volatility, GARCH-lite, log-normal model |
| §2 | Social physics — opinion dynamics, voter model, Fokker-Planck |
| §3 | Calculus rush — derivatives, integrals, optimization, vector calc (SymPy sprint) |
| §4 | Crypto rules database — SQLite schema, rule engine, backtesting |
| §5 | Military-backed currency — Petrodollar, reserve theory, commodity-backed crypto |

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import sqlite3, json, warnings
from scipy import stats, optimize
from datetime import datetime, timedelta
warnings.filterwarnings('ignore')
np.random.seed(2026)
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math
print('Price - Social - Calculus - Crypto - Military loading')

---
## §1 📈 Price History — Returns · Volatility · GARCH-Lite · Log-Normal

**Log-return:** $r_t = \ln(P_t / P_{t-1})$ — additive, ~normally distributed, scale-invariant.

**Geometric Brownian Motion (GBM):**
$$dS = \mu S\,dt + \sigma S\,dW_t \implies S_t = S_0 \exp\!\left[\left(\mu - \tfrac{\sigma^2}{2}\right)t + \sigma W_t\right]$$

**GARCH(1,1):** volatility clusters — $\sigma_t^2 = \omega + \alpha r_{t-1}^2 + \beta \sigma_{t-1}^2$

**Fat tails:** real returns have kurtosis > 3 (leptokurtic) — GBM underestimates crash probability.

In [ ]:
# §1 — Price history analysis

# §1.1 Synthetic multi-asset price history (GBM)
T_days = 252 * 4   # 4 years of trading days
dt_yr  = 1/252

assets = {
    'BTC':   {'mu': 0.80,  'sigma': 0.85, 'S0': 30_000},
    'ETH':   {'mu': 0.65,  'sigma': 0.95, 'S0': 1_800},
    'Gold':  {'mu': 0.08,  'sigma': 0.15, 'S0': 1_850},
    'SP500': {'mu': 0.12,  'sigma': 0.18, 'S0': 4_200},
    'Corn':  {'mu': 0.03,  'sigma': 0.22, 'S0': 6.50},   # $/bushel
}

dates   = [datetime(2022,1,1) + timedelta(days=i) for i in range(T_days)]
prices  = {}
returns = {}
for name, p in assets.items():
    dW   = np.random.randn(T_days) * np.sqrt(dt_yr)
    logr = (p['mu'] - 0.5*p['sigma']**2)*dt_yr + p['sigma']*dW
    S    = p['S0'] * np.exp(np.cumsum(logr))
    S    = np.insert(S[:-1], 0, p['S0'])
    prices[name]  = S
    returns[name] = np.diff(np.log(S))

# §1.2 GARCH(1,1) volatility estimate
def garch11(r, omega=1e-6, alpha=0.09, beta=0.90):
    sig2 = np.zeros(len(r))
    sig2[0] = np.var(r)
    for t in range(1, len(r)):
        sig2[t] = omega + alpha*r[t-1]**2 + beta*sig2[t-1]
    return np.sqrt(sig2) * np.sqrt(252)   # annualized vol

btc_r    = returns['BTC']
garch_vol= garch11(btc_r)
roll_vol = np.array([btc_r[max(0,i-20):i].std()*np.sqrt(252)
                     for i in range(1,len(btc_r)+1)])

print('§1.2 BTC GARCH(1,1) vs rolling vol (30-day avg):')
print(f'  GARCH mean vol:  {garch_vol.mean():.1%}')
print(f'  Rolling mean vol:{roll_vol.mean():.1%}')

# §1.3 Return distribution stats
print('\n§1.3 Return distribution:')
for name, r in returns.items():
    k  = stats.kurtosis(r)
    sk = stats.skew(r)
    sr = r.mean()/r.std() * np.sqrt(252)   # annualized Sharpe
    print(f'  {name:6s}: kurtosis={k:5.2f}  skew={sk:+.3f}  Sharpe={sr:+.2f}')

# §1.4 Value at Risk (VaR) and CVaR
alpha_var = 0.05
print(f'\n§1.4 VaR/CVaR at {alpha_var:.0%} (1-day, $1M position):')
for name, r in returns.items():
    var  = np.percentile(r, alpha_var*100)
    cvar = r[r <= var].mean()
    print(f'  {name:6s}: VaR={var:.3%}  CVaR={cvar:.3%}  '
          f'($1M VaR=${-var*1e6:,.0f}  CVaR=${-cvar*1e6:,.0f})')

# §1.5 Correlation matrix
ret_mat = np.array([returns[n] for n in assets]).T[1:]   # align lengths
corr    = np.corrcoef(ret_mat.T)

# ── Plots ─────────────────────────────────────────────────────
fig = plt.figure(figsize=(16,10))
gs  = gridspec.GridSpec(3,4,fig,hspace=0.45,wspace=0.32)

# Normalized price history
ax1 = fig.add_subplot(gs[0,:2])
for name,S in prices.items():
    ax1.plot(S/S[0], lw=1.5, label=name)
ax1.set_yscale('log')
ax1.set_xlabel('Trading day'); ax1.set_ylabel('Normalized price (log)')
ax1.set_title('§1.1 GBM price history (4yr)\nnormalized, log scale')
ax1.legend(fontsize=8)
ax1.axhline(1,color='gray',lw=0.7,ls='--')

# BTC with GARCH vol
ax2 = fig.add_subplot(gs[0,2:])
ax2b= ax2.twinx()
ax2.plot(prices['BTC'], 'royalblue', lw=1.5, label='BTC price')
ax2b.plot(garch_vol[1:], 'red', lw=1.2, alpha=0.7, label='GARCH vol')
ax2b.plot(roll_vol,      'orange', lw=1.2, alpha=0.6, ls='--', label='Roll vol (20d)')
ax2.set_ylabel('BTC ($)', color='royalblue')
ax2b.set_ylabel('Ann. vol', color='red')
ax2.set_title('§1.2 BTC price + GARCH(1,1)\nvolatility clustering')
lines1,labs1 = ax2.get_legend_handles_labels()
lines2,labs2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2,labs1+labs2,fontsize=7)

# Return distributions
ax3 = fig.add_subplot(gs[1,:2])
x_norm = np.linspace(-0.15, 0.15, 300)
for name,r in returns.items():
    ax3.hist(r, bins=80, alpha=0.35, density=True, label=name)
ax3.plot(x_norm, stats.norm.pdf(x_norm, 0, 0.03), 'k--', lw=2, label='N(0,3%)')
ax3.set_xlim(-0.15,0.15)
ax3.set_xlabel('Daily log-return'); ax3.set_ylabel('Density')
ax3.set_title('§1.3 Return distributions\n(fat tails vs Gaussian)')
ax3.legend(fontsize=7)

# QQ plot BTC
ax4 = fig.add_subplot(gs[1,2])
stats.probplot(returns['BTC'], dist='norm', plot=ax4)
ax4.set_title('§1.3 BTC Q-Q plot\n(vs normal — fat tails visible)')

# Correlation matrix
ax5 = fig.add_subplot(gs[1,3])
im5 = ax5.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax5.set_xticks(range(len(assets))); ax5.set_xticklabels(assets.keys(), rotation=30,fontsize=8)
ax5.set_yticks(range(len(assets))); ax5.set_yticklabels(assets.keys(), fontsize=8)
for i in range(len(assets)):
    for j in range(len(assets)):
        ax5.text(j,i,f'{corr[i,j]:.2f}',ha='center',va='center',fontsize=7,
                 color='white' if abs(corr[i,j])>0.5 else 'black')
plt.colorbar(im5, ax=ax5)
ax5.set_title('§1.5 Asset\ncorrelation matrix')

# VaR waterfall
ax6 = fig.add_subplot(gs[2,:2])
for name, r in returns.items():
    sorted_r = np.sort(r)
    ax6.plot(np.linspace(0,1,len(sorted_r)), sorted_r, lw=2, label=name)
ax6.axhline(0,color='gray',lw=0.5)
ax6.axvline(0.05,color='red',lw=1.5,ls='--',label='5% VaR cutoff')
ax6.set_xlabel('Quantile'); ax6.set_ylabel('Daily return')
ax6.set_title('§1.4 Return CDF\n(VaR at 5th percentile)')
ax6.legend(fontsize=7)

# Sharpe vs volatility scatter
ax7 = fig.add_subplot(gs[2,2:])
vols_  = [returns[n].std()*np.sqrt(252) for n in assets]
ret_   = [(returns[n].mean()*252) for n in assets]
sharpe_= [r/v for r,v in zip(ret_,vols_)]
colors_= ['orange','purple','gold','steelblue','green']
for i,(name,v,r,s) in enumerate(zip(assets,vols_,ret_,sharpe_)):
    ax7.scatter(v,r,s=abs(s)*300,c=colors_[i],edgecolors='#333',zorder=5)
    ax7.text(v+0.01,r+0.01,f'{name}\nSR={s:.2f}',fontsize=8)
ax7.set_xlabel('Annualized vol'); ax7.set_ylabel('Annualized return')
ax7.set_title('§1 Risk-return scatter\n(bubble size = |Sharpe|)')
ax7.axhline(0,color='gray',lw=0.5); ax7.axvline(0,color='gray',lw=0.5)
# Efficient frontier curve
vols_ef = np.linspace(0.05,1.2,200)
ax7.plot(vols_ef, 0.12 + 0.5*(vols_ef-0.18), 'k--', lw=1.5, alpha=0.4,
         label='Capital market line')
ax7.legend(fontsize=8)

plt.suptitle('§1: Price History — GBM · GARCH · Fat Tails · VaR/CVaR · Correlation',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §2 🌐 Social Physics — Opinion Dynamics · Voter Model · Fokker-Planck

**Voter model:** agents on a lattice copy a random neighbor's opinion.
Consensus time scales as $T \sim N \ln N$ on complete graph.

**Deffuant bounded-confidence model:** agent $i$ updates toward agent $j$
only if $|x_i - x_j| < \epsilon$ (bounded confidence):
$$x_i \leftarrow x_i + \mu(x_j - x_i), \quad x_j \leftarrow x_j + \mu(x_i - x_j)$$

**Mean-field / Fokker-Planck:**
$$\frac{\partial \rho}{\partial t} = -\frac{\partial}{\partial x}[A(x)\rho] + \frac{D}{2}\frac{\partial^2 \rho}{\partial x^2}$$
where $A(x) = -\gamma x + h$ (drift toward mean + external field $h$) models media influence.

**Connection to markets:** price consensus = opinion consensus with infinite agents and
continuous updating. BTC price *is* the social mean-field of crypto belief.

In [ ]:
# §2 — Social physics simulations

# §2.1 Voter model on complete graph
print('§2.1 Voter model (N=200, complete graph):')
N_voter  = 200
n_steps  = 10_000
# Initial opinions: 60/40 split
opinions = np.array([1]*120 + [-1]*80, dtype=float)
np.random.shuffle(opinions)

frac_history = [opinions.mean()]
for t in range(n_steps):
    i = np.random.randint(N_voter)
    j = np.random.randint(N_voter)
    opinions[i] = opinions[j]
    if t % 10 == 0:
        frac_history.append(opinions.mean())

print(f'  Start: frac+1 = 0.60, End: frac+1 = {(opinions==1).mean():.3f}')
print(f'  Consensus reached: {abs(opinions.mean()) > 0.98}')

# §2.2 Deffuant bounded-confidence model
def deffuant(N=500, epsilon=0.3, mu=0.5, steps=5000, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, N)   # opinions in [0,1]
    history = [x.copy()]
    for _ in range(steps):
        i,j = rng.integers(0,N,2)
        if abs(x[i]-x[j]) < epsilon:
            diff     = x[j]-x[i]
            x[i]    += mu*diff
            x[j]    -= mu*diff
        if _ % 50 == 0:
            history.append(x.copy())
    return x, history

print('\n§2.2 Deffuant model results:')
for eps in [0.1, 0.2, 0.3, 0.5]:
    xf, _ = deffuant(epsilon=eps)
    clusters = []
    sorted_x = np.sort(xf)
    cl_start = sorted_x[0]
    for xi in sorted_x[1:]:
        if xi - cl_start > 0.05: cl_start = xi; clusters.append(xi)
    n_clusters = len(clusters)+1
    print(f'  eps={eps:.1f}: ~{n_clusters} opinion clusters')

xf_03, hist_03 = deffuant(epsilon=0.3, steps=8000)
xf_01, hist_01 = deffuant(epsilon=0.1, steps=8000)

# §2.3 Fokker-Planck simulation (finite differences)
print('\n§2.3 Fokker-Planck opinion drift:')
Nx   = 200
x_fp = np.linspace(-3, 3, Nx)
dx   = x_fp[1]-x_fp[0]
dt_fp= 0.005
gamma= 1.0   # mean reversion
D_fp = 0.3   # noise / social diffusion
h_fp = 0.5   # external field (media bias)

# Initial condition: bimodal distribution (polarized society)
rho  = (np.exp(-4*(x_fp+1.2)**2) + np.exp(-4*(x_fp-1.2)**2))
rho /= rho.sum()*dx

snapshots = {0: rho.copy()}
for step in range(1, 3001):
    A      = -gamma*x_fp + h_fp   # drift
    # Upwind finite differences
    flux_r = A*rho - D_fp/(2*dx)*(np.roll(rho,-1)-rho)
    flux_l = np.roll(flux_r,1)
    diff   = D_fp/(2*dx**2)*(np.roll(rho,-1)-2*rho+np.roll(rho,1))
    drho   = -(flux_r-flux_l)/dx + diff
    rho   += dt_fp*drho
    rho    = np.maximum(rho,0); rho /= (rho.sum()*dx)
    if step in [200,800,2000,3000]:
        snapshots[step] = rho.copy()

mean_final = (x_fp*rho*dx).sum()
print(f'  Final opinion mean: {mean_final:.3f}  (external field h={h_fp} pushed right)')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# Voter model trajectory
ax1 = axes[0][0]
ax1.plot(np.arange(len(frac_history))*10, frac_history, 'royalblue',lw=1.5)
ax1.axhline(1,color='green',ls='--',lw=1.2,label='+1 consensus')
ax1.axhline(-1,color='red',ls='--',lw=1.2,label='-1 consensus')
ax1.axhline(0,color='gray',lw=0.5)
ax1.set_xlabel('Timestep'); ax1.set_ylabel('Mean opinion')
ax1.set_title('§2.1 Voter model\nN=200 complete graph')
ax1.legend(fontsize=8)

# Deffuant time evolution (scatter)
ax2 = axes[0][1]
steps_shown = min(len(hist_03), 50)
step_inds   = np.linspace(0, len(hist_03)-1, steps_shown, dtype=int)
for si in step_inds:
    alpha = 0.15 + 0.85*(si/len(hist_03))
    ax2.scatter([si]*len(hist_03[si]), hist_03[si], s=0.5,
                c=[[1-alpha,0.3,alpha]], alpha=0.3)
ax2.set_xlabel('Time step (x50)'); ax2.set_ylabel('Opinion value [0,1]')
ax2.set_title('§2.2 Deffuant model eps=0.3\n(opinion clustering over time)')

# Final opinion histograms
ax3 = axes[0][2]
ax3.hist(xf_03, bins=40, alpha=0.6, density=True, label='eps=0.3 (2 clusters)')
ax3.hist(xf_01, bins=40, alpha=0.6, density=True, label='eps=0.1 (fragmented)')
ax3.set_xlabel('Final opinion'); ax3.set_ylabel('Density')
ax3.set_title('§2.2 Deffuant final distributions\nbounded confidence comparison')
ax3.legend(fontsize=8)

# Fokker-Planck evolution
ax4 = axes[1][0]
colors_fp = plt.cm.viridis(np.linspace(0,1,len(snapshots)))
for (step,rho_s),col in zip(snapshots.items(),colors_fp):
    ax4.plot(x_fp, rho_s, color=col, lw=2, label=f't={step*0.005:.1f}')
ax4.axvline(h_fp/gamma, color='red', ls='--', lw=1.5, label=f'Equil x*={h_fp/gamma:.1f}')
ax4.set_xlabel('Opinion x'); ax4.set_ylabel('Density rho(x,t)')
ax4.set_title(f'§2.3 Fokker-Planck\ngamma={gamma} D={D_fp} h={h_fp} (media bias)')
ax4.legend(fontsize=7)

# Phase diagram: polarization vs epsilon (Deffuant)
ax5 = axes[1][1]
eps_range = np.linspace(0.05, 0.55, 20)
polariz   = []
for eps in eps_range:
    xf_e, _ = deffuant(epsilon=eps, N=300, steps=3000)
    # Polarization = std of final opinions
    polariz.append(xf_e.std())
ax5.plot(eps_range, polariz, 'o-', color='purple', lw=2, ms=5)
ax5.axvline(0.5, color='red', ls='--', lw=1.5, label='eps=0.5 (full consensus)')
ax5.set_xlabel('Confidence bound epsilon')
ax5.set_ylabel('Polarization (std of final opinions)')
ax5.set_title('§2.2 Phase diagram:\nPolarization vs confidence bound')
ax5.legend(fontsize=8)

# Social physics → price connection
ax6 = axes[1][2]
# Opinion dynamics with two groups: HODLers vs sellers
N_agents = 1000
opinion  = np.random.randn(N_agents)   # N(0,1) initial beliefs
price_sim= [30000.0]
for t in range(252):
    # Mean-field update with noise + price feedback
    mean_op  = opinion.mean()
    price_t  = price_sim[-1]
    # Agents update beliefs: drift toward crowd + random shock
    opinion += 0.05*(mean_op - opinion) + 0.2*np.random.randn(N_agents)
    # Price = exp of mean belief (log-normal mapping)
    new_price= price_t * np.exp(0.01*opinion.mean() + 0.03*np.random.randn())
    price_sim.append(max(100, new_price))
ax6.plot(price_sim, 'orange', lw=2)
ax6.set_xlabel('Day'); ax6.set_ylabel('BTC price ($)')
ax6.set_title('§2 Social mean-field → price\n(opinion consensus drives BTC)')
ax6b = ax6.twinx()
# Plot mean opinion
op_means = [0] + [np.random.randn()*0.05 for _ in range(252)]
ax6b.plot(np.cumsum(op_means)*5, 'royalblue', lw=1.5, alpha=0.7, label='Mean opinion')
ax6b.set_ylabel('Mean opinion', color='royalblue')

plt.suptitle('§2: Social Physics — Voter Model · Deffuant · Fokker-Planck · Price Emergence',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §3 ∫ Calculus Rush — Symbolic Sprint with SymPy

**Five pillars of calculus** (one cell each, full SymPy + matplotlib):

1. **Limits** — $\varepsilon$-$\delta$, L'Hôpital, asymptotic behavior
2. **Derivatives** — chain rule, implicit, partial, Jacobian, Hessian
3. **Integrals** — FTC, substitution, integration by parts, improper
4. **Series** — Taylor/Maclaurin, radius of convergence, Fourier series
5. **Vector calculus** — gradient, divergence, curl, Stokes'/Green's theorem

In [ ]:
# §3 — Calculus rush (SymPy sprint)

x,y,z,t,n,a,b,k = sp.symbols('x y z t n a b k', real=True)
eps, delta       = sp.symbols('epsilon delta', positive=True)
f, g             = sp.Function('f'), sp.Function('g')

print('='*55)
print('§3 CALCULUS RUSH — SymPy sprint')
print('='*55)

# ── 3.1 Limits ────────────────────────────────────────────────
print('\n[3.1 Limits]')
limits_table = [
    (sp.sin(x)/x, x, 0,   '(sinc)'),
    ((1+1/n)**n,  n, sp.oo,'(Euler e)'),
    ((sp.exp(x)-1)/x, x, 0,'(exp diff-quot)'),
    (x*sp.log(x), x, 0, sp.S.One, '(x*ln x as x->0+)'),
    (sp.tan(x)-sp.sin(x), x, 0, '(LHopital demo)'),
]
for entry in limits_table:
    if len(entry)==4 and isinstance(entry[3],str):
        expr,var,pt,note = entry
        L = sp.limit(expr,var,pt)
        print(f'  lim_{var}->{pt} {expr} = {L}  {note}')
    elif len(entry)==5:
        expr,var,pt,side,note = entry
        L = sp.limit(expr,var,pt,side)
        print(f'  lim_{var}->{pt}+ {expr} = {L}  {note}')

# ── 3.2 Derivatives ───────────────────────────────────────────
print('\n[3.2 Derivatives]')
funcs_d = {
    'sin(x^3)*exp(-x^2)': sp.sin(x**3)*sp.exp(-x**2),
    'ln(1+x^2)/x':        sp.log(1+x**2)/x,
    'x^x (=exp(x ln x))': x**x,
    '(x^2+y^2)^(3/2)  dx':sp.Rational(3,2),  # placeholder
    'arctan(y/x) dy':      sp.atan(y/x),
}
for name, expr in funcs_d.items():
    if name.endswith('dx)'):
        fxy = (x**2+y**2)**sp.Rational(3,2)
        dx_ = sp.diff(fxy,x)
        dy_ = sp.diff(fxy,y)
        print(f'  d/dx (x^2+y^2)^(3/2) = {sp.simplify(dx_)}')
        print(f'  d/dy (x^2+y^2)^(3/2) = {sp.simplify(dy_)}')
    elif 'arctan' in name:
        print(f'  d/dy arctan(y/x) = {sp.diff(expr,y)} = {sp.simplify(sp.diff(expr,y))}')
    else:
        dd = sp.diff(expr,x)
        print(f'  d/dx {name} = {sp.simplify(dd)}')

# Jacobian & Hessian
F_vec = sp.Matrix([x**2+y*sp.sin(z), x*y + sp.exp(z)])
J = F_vec.jacobian([x,y,z])
print(f'\n  Jacobian of F=(x^2+y*sin(z), x*y+e^z):')
sp.pprint(J, use_unicode=False)

H_func = x**4 + y**4 - 4*x*y
H = sp.hessian(H_func, [x,y])
print(f'\n  Hessian of x^4+y^4-4xy:')
sp.pprint(H, use_unicode=False)
crits = sp.solve([sp.diff(H_func,x), sp.diff(H_func,y)], [x,y])
print(f'  Critical points: {crits}')

# ── 3.3 Integrals ─────────────────────────────────────────────
print('\n[3.3 Integrals]')
integrals_tbl = [
    (x**n,              x, '  power rule: x^n'),
    (sp.sin(x)*sp.exp(x), x, '  IBP: sin(x)e^x'),
    (1/(1+x**2),        x, '  -> arctan'),
    (sp.exp(-x**2),     (x,-sp.oo,sp.oo), 'improper: Gaussian'),
    (sp.log(x),         (x,1,sp.E),       'definite: ln(x) 1->e'),
    (x**2 * sp.exp(-a*x),(x,0,sp.oo),     'Laplace-type'),
]
for entry in integrals_tbl:
    if len(entry)==3 and not isinstance(entry[1],tuple):
        expr,var,note = entry
        I = sp.integrate(expr,var)
        print(f'  {note:35s} = {I}')
    else:
        expr,bounds,note = entry
        I = sp.integrate(expr, bounds)
        Is = sp.simplify(I)
        print(f'  {note:35s} = {Is}')

# FTC demo
F_antideriv = sp.integrate(sp.cos(x**2), x)
print(f'\n  d/dx [integral of cos(x^2) dx] = {sp.diff(F_antideriv,x)}  (FTC)')

# ── 3.4 Series ────────────────────────────────────────────────
print('\n[3.4 Series]')
series_funcs = [
    (sp.sin(x),     x, 0, 7,  'sin(x)'),
    (sp.exp(x),     x, 0, 6,  'e^x'),
    (1/(1-x),       x, 0, 6,  '1/(1-x) geometric'),
    (sp.log(1+x),   x, 0, 7,  'ln(1+x)'),
    (sp.cos(x),     x, 0, 7,  'cos(x)'),
]
for expr,var,pt,order,name in series_funcs:
    s = sp.series(expr,var,pt,order)
    print(f'  {name:22s}: {s}')

# Fourier series (square wave)
print('\n  Fourier series of square wave (first 5 harmonics):')
fourier_terms = [sp.Rational(4,1)/sp.pi * sp.sin((2*k+1)*x)/(2*k+1) for k in range(5)]
f_sq = sum(fourier_terms)
print(f'  f(x) = {f_sq}')

# Radius of convergence: ratio test on power series
a_n = sp.Function('a')
print('\n  Ratio test R = 1/limsup |a_{n+1}/a_n|:')
for expr_an,name in [(1/sp.factorial(n),'1/n!'), (x**n/(n**2),'x^n/n^2'), (n**n,'n^n')]:
    ratio = sp.limit(sp.Abs(expr_an.subs(n,n+1)/expr_an), n, sp.oo)
    R = sp.simplify(1/ratio) if ratio != 0 else sp.oo
    print(f'  a_n={name:12s}  |a_{{n+1}}/a_n|->  {ratio}  R={R}')

# ── 3.5 Vector calculus ────────────────────────────────────────
print('\n[3.5 Vector calculus]')
Fx = sp.Function('Fx')(x,y,z)
Fy = sp.Function('Fy')(x,y,z)
Fz = sp.Function('Fz')(x,y,z)

# Specific vector field: F = (x^2 y, y^2 z, z^2 x)
Fx_v = x**2*y; Fy_v = y**2*z; Fz_v = z**2*x

div_F  = sp.diff(Fx_v,x) + sp.diff(Fy_v,y) + sp.diff(Fz_v,z)
curl_F = sp.Matrix([
    sp.diff(Fz_v,y)-sp.diff(Fy_v,z),
    sp.diff(Fx_v,z)-sp.diff(Fz_v,x),
    sp.diff(Fy_v,x)-sp.diff(Fx_v,y)])
phi    = sp.exp(-x**2-y**2-z**2)
grad_phi= sp.Matrix([sp.diff(phi,x), sp.diff(phi,y), sp.diff(phi,z)])
lapl   = sp.diff(phi,x,2)+sp.diff(phi,y,2)+sp.diff(phi,z,2)
lapl_s = sp.simplify(lapl)

print(f'  F = (x^2 y, y^2 z, z^2 x)')
print(f'  div F = {div_F}')
print(f'  curl F = {curl_F.T}')
print(f'\n  phi = exp(-(x^2+y^2+z^2))')
print(f'  grad phi = {grad_phi.T}')
print(f'  Laplacian phi = {lapl_s}')

# Green s theorem numeric check: line integral = area
theta_g= np.linspace(0, 2*np.pi, 10000)
r_g    = 2.0   # circle radius
cx,cy  = r_g*np.cos(theta_g), r_g*np.sin(theta_g)
dx_g   = np.diff(cx,append=cx[0])
dy_g   = np.diff(cy,append=cy[0])
line_int = 0.5*np.sum(cx*dy_g - cy*dx_g)   # should = pi*r^2
print(f'\n  Green\'s theorem: closed line integral (circle r={r_g}):')
print(f'  (1/2)∮(x dy - y dx) = {line_int:.6f}  vs pi*r^2 = {np.pi*r_g**2:.6f}')

In [ ]:
# §3 Calculus rush — visualization

fig, axes = plt.subplots(2,4,figsize=(16,8))

# Taylor series convergence
x_num= np.linspace(-3,3,400)
ax1  = axes[0][0]
ax1.plot(x_num, np.sin(x_num), 'k', lw=3, label='sin(x) exact')
for order,col in [(1,'red'),(3,'orange'),(5,'green'),(7,'blue')]:
    coeffs = [(-1)**(k//2)/np.math.factorial(k) if k%2==1 else 0
              for k in range(order+1)]
    y_t = sum(c*x_num**k for k,c in enumerate(coeffs))
    ax1.plot(x_num, np.clip(y_t,-2.5,2.5), color=col, lw=1.5, ls='--', label=f'O({order})')
ax1.set_ylim(-2.5,2.5); ax1.legend(fontsize=7)
ax1.set_title('§3.4 Taylor series\nsin(x) convergence')
ax1.axhline(0,color='gray',lw=0.5); ax1.axvline(0,color='gray',lw=0.5)

# Fourier square wave
x_f   = np.linspace(-np.pi, np.pi, 1000)
ax2   = axes[0][1]
sq_wave= np.sign(np.sin(x_f))
ax2.plot(x_f, sq_wave, 'k', lw=2.5, label='Square wave',alpha=0.6)
running = np.zeros_like(x_f)
for k_val,col in enumerate(['red','orange','green','blue','purple']):
    running += 4/np.pi * np.sin((2*k_val+1)*x_f)/(2*k_val+1)
    ax2.plot(x_f, running, color=col, lw=1.5, label=f'{2*k_val+1} harmonics')
ax2.legend(fontsize=7); ax2.set_title('§3.4 Fourier series\nsquare wave convergence')

# Vector field: F=(x^2 y, y^2 z) in xy-plane (z=0)
X2,Y2 = np.meshgrid(np.linspace(-2,2,12), np.linspace(-2,2,12))
Fxn   = X2**2*Y2;  Fyn = Y2**2*0   # z=0 -> Fy_v=0
U2    = Fxn/np.maximum(np.sqrt(Fxn**2+Fyn**2),1e-3)
V2    = Fyn/np.maximum(np.sqrt(Fxn**2+Fyn**2),1e-3)
ax3   = axes[0][2]
ax3.quiver(X2,Y2,U2,V2,np.sqrt(Fxn**2),cmap='plasma',scale=15,width=0.005)
ax3.set_title('§3.5 Vector field F=(x^2 y,0)\ndivergence + curl')
ax3.set_xlabel('x'); ax3.set_ylabel('y')

# Gradient descent on H_func = x^4+y^4-4xy
X3,Y3 = np.meshgrid(np.linspace(-2,2,100), np.linspace(-2,2,100))
Z3    = X3**4+Y3**4-4*X3*Y3
ax4   = axes[0][3]
cs    = ax4.contourf(X3,Y3,Z3, levels=40, cmap='RdYlGn_r')
ax4.contour(X3,Y3,Z3, levels=20, colors='white', lw=0.5, alpha=0.3)
# Gradient descent path
pt = np.array([1.5,1.5])
path = [pt.copy()]
for _ in range(200):
    gx = 4*pt[0]**3 - 4*pt[1]
    gy = 4*pt[1]**3 - 4*pt[0]
    pt -= 0.02*np.array([gx,gy])
    path.append(pt.copy())
path = np.array(path)
ax4.plot(path[:,0],path[:,1],'white',lw=2,label='Gradient descent')
ax4.plot(*path[0],'wo',ms=8); ax4.plot(*path[-1],'r*',ms=12,label='Min')
ax4.legend(fontsize=8); ax4.set_title('§3.2/3.5 Gradient descent\nx^4+y^4-4xy')
plt.colorbar(cs,ax=ax4)

# Derivative visualization
x_dv = np.linspace(-2,2,400)
f_dv = np.exp(-x_dv**2)*np.sin(3*x_dv)
df   = np.gradient(f_dv, x_dv)
d2f  = np.gradient(df,   x_dv)
ax5  = axes[1][0]
ax5.plot(x_dv,f_dv, 'royalblue',lw=2.5,label='f(x)=e^{-x^2}sin(3x)')
ax5.plot(x_dv,df,   'green',    lw=2,  label="f'(x)")
ax5.plot(x_dv,d2f,  'red',      lw=2,  label="f''(x)")
ax5.axhline(0,color='gray',lw=0.5)
ax5.legend(fontsize=8); ax5.set_title("§3.2 f, f', f''\ne^{-x^2}sin(3x)")

# Integration convergence
ax6 = axes[1][1]
Ns_int = [4,8,16,32,64,128,256]
true_I = np.sqrt(np.pi)   # int e^{-x^2} dx = sqrt(pi)
riemann_errs = []
trap_errs    = []
simp_errs    = []
for N_i in Ns_int:
    xi = np.linspace(-5,5,N_i+1); hi = xi[1]-xi[0]
    fi = np.exp(-xi**2)
    riemann_errs.append(abs(hi*fi[:-1].sum() - true_I))
    trap_errs.append(   abs(np.trapz(fi,xi)  - true_I))
    # Simpson
    if N_i%2==0:
        simp_errs.append(abs(hi/3*(fi[0]+4*fi[1:-1:2].sum()+2*fi[2:-2:2].sum()+fi[-1])-true_I))
    else:
        simp_errs.append(trap_errs[-1])
ax6.loglog(Ns_int,riemann_errs,'o-',label='Riemann')
ax6.loglog(Ns_int,trap_errs,   's-',label='Trapezoid  (O(h^2))')
ax6.loglog(Ns_int,simp_errs,   '^-',label="Simpson's  (O(h^4))")
ax6.set_xlabel('N intervals'); ax6.set_ylabel('|error|')
ax6.set_title('§3.3 Integration error\nconvergence rate')
ax6.legend(fontsize=8)

# Implicit curve
ax7 = axes[1][2]
ax7.contour(X3,Y3,X3**3-Y3**2-X3*Y3, levels=[0], colors='royalblue',lw=3)
ax7.contour(X3,Y3,X3**2+Y3**2,       levels=[1,2,4], colors='red')
ax7.set_title('§3.2 Implicit curves\nx^3-y^2-xy=0 (blue)\ncircles (red)')
ax7.set_xlabel('x'); ax7.set_ylabel('y')

# Stokes theorem numeric (curl F over surface = line integral)
theta_s = np.linspace(0, 2*np.pi, 5000)
R_s     = 1.5
cx_s,cy_s = R_s*np.cos(theta_s), R_s*np.sin(theta_s)
dx_s = np.diff(cx_s, append=cx_s[0])
dy_s = np.diff(cy_s, append=cy_s[0])
# F = (y, -x): line integral = -2*area
line_s = np.sum(cy_s*dx_s + (-cx_s)*dy_s)
area_s = np.pi*R_s**2
# curl F = dFy/dx - dFx/dy = -1 - 1 = -2
expected_s = -2*area_s
ax8 = axes[1][3]
theta_plot = np.linspace(0,2*np.pi,300)
ax8.plot(R_s*np.cos(theta_plot),R_s*np.sin(theta_plot),'b',lw=2.5,label=f'Contour r={R_s}')
ax8.fill(R_s*np.cos(theta_plot),R_s*np.sin(theta_plot),alpha=0.15,color='blue')
ax8.set_aspect('equal')
ax8.set_title(f"§3.5 Stokes' theorem\nF=(y,-x): ∮F·dr={line_s:.4f}\n2*curl*area={expected_s:.4f}")
ax8.legend(fontsize=8)
ax8.axhline(0,color='gray',lw=0.5); ax8.axvline(0,color='gray',lw=0.5)

plt.suptitle('§3: Calculus Rush — Taylor · Fourier · Gradient · Integration · Vector Calc',
             fontsize=12,fontweight='bold')
plt.tight_layout(); plt.show()

---
## §4 🗄️ Crypto Rules Database — SQLite Schema · Rule Engine · Backtesting

**Design:** store trading rules as rows — condition (signal), action (buy/sell/hold),
asset, parameters. A rule engine queries the DB each tick and fires matching rules.

**Schema:**
```sql
CREATE TABLE rules (
    id INTEGER PRIMARY KEY,
    name TEXT, asset TEXT,
    condition TEXT,           -- 'price_above', 'rsi_below', 'vol_spike', ...
    threshold REAL,
    action TEXT,              -- 'buy', 'sell', 'hold'
    size_pct REAL,            -- fraction of portfolio
    active INTEGER DEFAULT 1,
    created_at TEXT
);
```

In [ ]:
# §4 — Crypto rules database + backtester

import sqlite3, json
from dataclasses import dataclass
from typing import List, Optional
import numpy as np

# §4.1 Create and populate rules DB
DB_PATH = ':memory:'   # in-memory for notebook; swap to file path for persistence

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur  = conn.cursor()

# Schema
cur.executescript('''
CREATE TABLE IF NOT EXISTS assets (
    symbol TEXT PRIMARY KEY,
    name   TEXT,
    sector TEXT
);

CREATE TABLE IF NOT EXISTS rules (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT NOT NULL,
    asset       TEXT NOT NULL REFERENCES assets(symbol),
    condition   TEXT NOT NULL,
    threshold   REAL,
    action      TEXT NOT NULL CHECK(action IN ("buy","sell","hold")),
    size_pct    REAL DEFAULT 0.10,
    active      INTEGER DEFAULT 1,
    priority    INTEGER DEFAULT 5,
    notes       TEXT,
    created_at  TEXT DEFAULT (datetime("now"))
);

CREATE INDEX IF NOT EXISTS idx_rules_asset  ON rules(asset);
CREATE INDEX IF NOT EXISTS idx_rules_active ON rules(active);

CREATE TABLE IF NOT EXISTS signals (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    ts         TEXT,
    rule_id    INTEGER REFERENCES rules(id),
    asset      TEXT,
    price      REAL,
    action     TEXT,
    executed   INTEGER DEFAULT 0
);

CREATE TABLE IF NOT EXISTS portfolio (
    asset    TEXT PRIMARY KEY,
    qty      REAL DEFAULT 0,
    avg_cost REAL DEFAULT 0,
    cash     REAL DEFAULT 100000
);
''')

# Seed assets
cur.executemany('INSERT OR IGNORE INTO assets VALUES (?,?,?)', [
    ('BTC',  'Bitcoin',  'Layer 1'),
    ('ETH',  'Ethereum', 'Layer 1 / Smart Contract'),
    ('SOL',  'Solana',   'Layer 1 High Throughput'),
    ('CORN', 'Corn Futures', 'Commodity'),
    ('USDC', 'USD Coin', 'Stablecoin'),
])

# Seed trading rules (the "rule book")
rules_seed = [
    ('RSI Oversold BTC',   'BTC', 'rsi_below',    30, 'buy',  0.15, 1, 9, 'Classic RSI oversold entry'),
    ('RSI Overbought BTC', 'BTC', 'rsi_above',    70, 'sell', 0.20, 1, 9, 'RSI overbought exit'),
    ('MA Cross Bull ETH',  'ETH', 'price_above_ma',0, 'buy',  0.10, 1, 7, 'Price crosses 20-day MA up'),
    ('Vol Spike Exit',     'BTC', 'vol_spike',     2.5,'sell', 0.25, 1, 8, 'Annualized vol >2.5x normal'),
    ('BTC Floor Buy',      'BTC', 'price_below',30000,'buy',  0.10, 1, 6, 'Accumulate below $30k'),
    ('Corn Harvest Sell',  'CORN','seasonal_peak', 9,  'sell', 0.50, 1, 5, 'Sep harvest sell pressure'),
    ('Stablecoin Refuge',  'USDC','market_fear',   75, 'buy',  0.30, 1, 10,'Flee to USDC on fear index >75'),
    ('ETH Dip Buy',        'ETH', 'drawdown_pct', 20,  'buy',  0.12, 0, 6, '20% drawdown (inactive)'),
]
cur.executemany(
    'INSERT INTO rules (name,asset,condition,threshold,action,size_pct,active,priority,notes) VALUES (?,?,?,?,?,?,?,?,?)',
    rules_seed
)
conn.commit()

# §4.2 Query and display rules
print('§4.1 Active trading rules:')
print(f'  {"ID":3s} {"Priority":8s} {"Asset":5s} {"Condition":20s} {"Threshold":10s} {"Action":5s} {"Size%":6s} {"Name"}')
print('  '+'-'*80)
for r in cur.execute('SELECT * FROM rules WHERE active=1 ORDER BY priority DESC, id'):
    print(f'  {r["id"]:3d} {r["priority"]:8d} {r["asset"]:5s} {r["condition"]:20s} '
          f'{str(r["threshold"] or ""):10s} {r["action"]:5s} {r["size_pct"]*100:5.0f}%  {r["name"]}')

# §4.3 Rule engine: evaluate rules against live data
@dataclass
class MarketData:
    asset: str; price: float; rsi: float
    vol: float; ma20: float; drawdown_pct: float
    fear_index: float; month: int

def evaluate_rules(conn, market: MarketData) -> List[dict]:
    '''Run active rules against current market snapshot.'''
    cur_ = conn.cursor()
    rules_ = cur_.execute(
        'SELECT * FROM rules WHERE asset=? AND active=1 ORDER BY priority DESC',
        (market.asset,)
    ).fetchall()
    signals = []
    for r in rules_:
        cond    = r['condition']
        thresh  = r['threshold']
        fired   = False
        if cond == 'rsi_below'       and market.rsi < thresh: fired = True
        elif cond == 'rsi_above'     and market.rsi > thresh: fired = True
        elif cond == 'price_below'   and market.price < thresh: fired = True
        elif cond == 'price_above'   and market.price > thresh: fired = True
        elif cond == 'price_above_ma'and market.price > market.ma20: fired = True
        elif cond == 'vol_spike'     and market.vol > thresh*0.6: fired = True
        elif cond == 'drawdown_pct'  and market.drawdown_pct > thresh: fired = True
        elif cond == 'market_fear'   and market.fear_index > thresh: fired = True
        elif cond == 'seasonal_peak' and market.month == int(thresh): fired = True
        if fired:
            signals.append({'rule_id':r['id'],'rule_name':r['name'],
                            'action':r['action'],'size_pct':r['size_pct']})
    return signals

# Test the rule engine
test_markets = [
    MarketData('BTC', 28500, rsi=28, vol=1.2, ma20=32000, drawdown_pct=12, fear_index=70, month=6),
    MarketData('BTC', 42000, rsi=72, vol=3.2, ma20=38000, drawdown_pct=2,  fear_index=45, month=3),
    MarketData('ETH', 2200,  rsi=55, vol=0.9, ma20=2100,  drawdown_pct=5,  fear_index=50, month=6),
    MarketData('CORN',6.80,  rsi=60, vol=0.3, ma20=6.50,  drawdown_pct=1,  fear_index=40, month=9),
]
print('\n§4.3 Rule engine evaluation:')
for md_ in test_markets:
    sigs = evaluate_rules(conn, md_)
    print(f'\n  {md_.asset} @ ${md_.price:,.0f}  RSI={md_.rsi}  vol={md_.vol:.1f}:')
    if sigs:
        for s in sigs:
            print(f'    SIGNAL: {s["action"].upper():4s} {s["size_pct"]*100:.0f}%  [{s["rule_name"]}]')
    else:
        print('    (no rules fired)')

# §4.4 Simple backtester
print('\n§4.4 Backtesting BTC rules (252 days):')
np.random.seed(42)
prices_bt = np.zeros(252); prices_bt[0] = 30000
for i in range(1,252):
    prices_bt[i] = prices_bt[i-1]*np.exp(
        (0.0002 + 0.0003*np.random.randn()) + 0.04*np.random.randn())

cash = 100_000.0; btc_held = 0.0
portfolio_val = [cash]
trades = 0

for i in range(20, len(prices_bt)):
    P   = prices_bt[i]
    r_w = np.log(prices_bt[i-20:i]/prices_bt[i-21:i-1])
    # Compute indicators
    delta_r = np.diff(prices_bt[i-14:i+1])
    gains   = delta_r[delta_r>0].sum(); losses = -delta_r[delta_r<0].sum()
    rsi_    = 100 - 100/(1 + (gains+1e-9)/(losses+1e-9))
    vol_    = r_w.std()*np.sqrt(252)
    ma20_   = prices_bt[i-20:i].mean()
    peak    = prices_bt[max(0,i-60):i].max()
    dd_     = (peak-P)/peak*100
    # RSI oversold -> buy
    if rsi_ < 30 and cash > P:
        qty = min(cash*0.15/P, cash/P); btc_held += qty; cash -= qty*P; trades+=1
    # RSI overbought -> sell
    elif rsi_ > 70 and btc_held > 0:
        cash += btc_held*0.20*P; btc_held -= btc_held*0.20; trades+=1
    # Vol spike -> reduce
    elif vol_ > 1.5 and btc_held > 0:
        cash += btc_held*0.25*P; btc_held -= btc_held*0.25; trades+=1
    portfolio_val.append(cash + btc_held*P)

final_val = portfolio_val[-1]
bh_val    = 100_000*(prices_bt[-1]/prices_bt[0])
print(f'  Rule-based strategy:  ${final_val:>12,.0f}  ({(final_val/1e5-1)*100:+.1f}%,  {trades} trades)')
print(f'  Buy-and-hold BTC:     ${bh_val:>12,.0f}  ({(bh_val/1e5-1)*100:+.1f}%)')

# CRUD: add a new rule
cur.execute('''INSERT INTO rules (name,asset,condition,threshold,action,size_pct,active,priority,notes)
               VALUES (?,?,?,?,?,?,?,?,?)''',
            ('Mean Reversion ETH','ETH','drawdown_pct',15,'buy',0.08,1,7,'Buy ETH -15% drawdown'))
conn.commit()
new_id = cur.lastrowid
print(f'\n§4 CRUD: inserted rule id={new_id}: Mean Reversion ETH')
print(f'  Total active rules: {cur.execute("SELECT COUNT(*) FROM rules WHERE active=1").fetchone()[0]}')
conn.close()

---
## §5 ⚔️ Military-Backed Currency — Petrodollar · Reserve Theory · Commodity Crypto

**Why does currency have value?** Three theories:

| Theory | Backing | Example |
|--------|---------|---------|
| **Commodity** | Gold/oil/grain — intrinsic value | Gold standard, WW2 Bretton Woods |
| **Fiat** | Legal decree + military enforcement | USD post-1971, any sovereign currency |
| **Hash rate** | Proof-of-work energy expenditure | Bitcoin — "digital energy" |

**Petrodollar system (1973):** Nixon closed gold window (1971) → Kissinger brokered
deal with Saudi Arabia: oil sold in USD globally → demand for USD is permanent →
USD hegemony backed by US military guaranteeing oil supply routes (Strait of Hormuz).

**"A currency needs a military"** — Triffin dilemma: the reserve currency issuer
must run deficits to supply global liquidity. This requires credibility = military
capability to enforce contracts globally.

**Crypto food/commodity backed:** a stablecoin pegged to a basket of commodities
(grain, energy, water) could decouple from military power — but requires:
- Audit oracle (Chainlink-type price feed from physical warehouses)
- Collateralization ratio > 1 (overcollateralized like DAI)
- Legal enforcement of redemption rights (back to sovereignty/military)

**Bitcoin's military analog = hash rate:** attacking Bitcoin requires >51% of
global hash rate — equivalent to $18B+ in ASICs. This IS a form of economic
deterrence analogous to military deterrence.

In [ ]:
# §5 — Military-backed currency models

# §5.1 Petrodollar system simulation
print('§5.1 Petrodollar recycling model:')

# Oil exporters receive USD -> invest in US Treasuries
oil_price_hist = np.array([3,4,4,12,14,32,28,18,20,22,25,35,55,72,95,60,80,110,90,105,
                            80,50,60,75,85,95,70,55,80,95,105,85])  # $/bbl 1970-2025 est
years_oil      = np.arange(1970, 2026, 53/32)[:len(oil_price_hist)]

# USD foreign reserves vs oil price
oil_export_M   = 30e6 * 365   # 30M bbl/day world export * 365
usd_revenue    = oil_price_hist * oil_export_M / 1e12   # $T/yr
# Petrodollar recycling: ~60% into US Treasuries
treasury_demand= usd_revenue * 0.60

print(f'  Peak oil revenue: ${usd_revenue.max():.1f}T/yr (at ${oil_price_hist.max():.0f}/bbl)')
print(f'  Est Treasury demand at peak: ${treasury_demand.max():.2f}T/yr')

# §5.2 Reserve currency share (simulated history)
years_res = np.arange(1945, 2026)
usd_share = 100/(1+np.exp(-0.15*(years_res-1950)))   # logistic rise
usd_share *= (1 - 0.003*(years_res-1970)*(years_res>1970))   # slow decline
usd_share  = np.clip(usd_share, 30, 75)
eur_share  = np.where(years_res>1999, 20*(1-np.exp(-0.15*(years_res-1999))), 0)
cny_share  = np.where(years_res>2000, 8*(1-np.exp(-0.1*(years_res-2000))), 0)
gold_share = np.where(years_res<1971, 100-usd_share, 0)
other      = np.clip(100-usd_share-eur_share-cny_share-gold_share, 0, 100)

# §5.3 Commodity-backed crypto model
print('\n§5.3 Commodity-backed stablecoin model:')
commodities = {
    'Wheat':   {'price_usd': 6.5,  'unit': 'bushel', 'weight': 0.20},
    'Corn':    {'price_usd': 6.8,  'unit': 'bushel', 'weight': 0.20},
    'Oil':     {'price_usd': 85.0, 'unit': 'barrel', 'weight': 0.30},
    'Nat Gas': {'price_usd': 2.8,  'unit': 'mmBTU',  'weight': 0.15},
    'Water':   {'price_usd': 0.05, 'unit': 'gallon',  'weight': 0.15},
}

# Basket value = weighted sum of normalized prices
basket_val = sum(c['price_usd']*c['weight'] for c in commodities.values())
print(f'  Commodity basket value: ${basket_val:.2f}')
print(f'  Collateral ratio 1.5x: ${basket_val*1.5:.2f} required per $1 token')

# Simulate stablecoin peg stability under commodity shock
np.random.seed(42)
T_stab  = 252
# Commodity price shocks (correlated: oil shock affects food)
shocks  = np.random.multivariate_normal(
    mean=[0]*5,
    cov=[[0.04,0.02,0.01,0.01,0.005],
         [0.02,0.04,0.01,0.01,0.005],
         [0.01,0.01,0.09,0.05,0.001],
         [0.01,0.01,0.05,0.06,0.001],
         [0.005,0.005,0.001,0.001,0.01]],
    size=T_stab
)
weights_arr = np.array([c['weight'] for c in commodities.values()])
prices_arr  = np.array([c['price_usd'] for c in commodities.values()])
# Simulate commodity price paths
comm_prices = np.zeros((T_stab,5))
comm_prices[0] = prices_arr
for i in range(1,T_stab):
    comm_prices[i] = comm_prices[i-1]*np.exp(0.0001 + 0.02*shocks[i])
basket_track = (comm_prices * weights_arr).sum(axis=1)
basket_norm  = basket_track / basket_track[0]   # normalized to 1
print(f'  Basket volatility (ann): {basket_norm.std()*np.sqrt(252):.1%}')
print(f'  vs BTC vol: ~80-90%  Gold vol: ~15%  USD CPI vol: ~3%')

# §5.4 Hash rate as deterrence
print('\n§5.4 Bitcoin hash rate deterrence:')
hashrate_EH = np.array([0.001,0.01,0.1,1,5,10,30,60,100,150,200,350,450,
                         380,320,500,580])  # EH/s 2009-2025 approx
years_hr    = np.arange(2009, 2026)
asic_cost_per_EH = 2e9   # $2B per EH/s to build (2024 estimate)
attack_cost = hashrate_EH * 0.51 * asic_cost_per_EH / 1e9  # $B
print(f'  2024 hash rate: ~{hashrate_EH[-1]:.0f} EH/s')
print(f'  51% attack cost: ~${attack_cost[-1]:.0f}B in ASICs alone')
print(f'  + electricity: ~${hashrate_EH[-1]*0.51*3e6/1e9:.1f}B/yr for power')
print(f'  Compare: US military budget 2024: ~$858B/yr')
print(f'  BTC deterrence: ${attack_cost[-1]:.0f}B upfront — comparable to a mid-tier military')

# ── Plots ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2,3,figsize=(16,9))

# Oil price + Petrodollar recycling
ax1 = axes[0][0]
ax1.fill_between(years_oil, oil_price_hist, alpha=0.4, color='gold')
ax1.plot(years_oil, oil_price_hist, 'darkgoldenrod', lw=2)
ax1.axvline(1971, color='red', ls='--', lw=1.5, label='Nixon shock (gold window)')
ax1.axvline(1973, color='orange', ls='--', lw=1.5, label='OPEC embargo')
ax1.axvline(1991, color='blue', ls='--', lw=1.5, label='Gulf War')
ax1.set_xlabel('Year'); ax1.set_ylabel('Oil price ($/bbl)')
ax1.set_title('§5.1 Oil price history\n+ Petrodollar inflection points')
ax1.legend(fontsize=7)

# Reserve currency share
ax2 = axes[0][1]
ax2.stackplot(years_res,
              usd_share, eur_share, cny_share, gold_share, other,
              labels=['USD','EUR','CNY','Gold (pre-71)','Other'],
              colors=['royalblue','gold','red','orange','gray'],
              alpha=0.8)
ax2.axvline(1971, color='white', lw=2, ls='--')
ax2.axvline(1999, color='white', lw=1.5, ls=':')
ax2.set_xlabel('Year'); ax2.set_ylabel('% of global reserves')
ax2.set_title('§5.2 Reserve currency share\n(USD dominance post-Bretton Woods)')
ax2.legend(fontsize=7, loc='upper left')

# Commodity basket stability
ax3 = axes[0][2]
names_cm = list(commodities.keys())
for i,name in enumerate(names_cm):
    ax3.plot(comm_prices[:,i]/comm_prices[0,i], lw=1.5, alpha=0.7, label=name)
ax3.plot(basket_norm, 'k', lw=3, label='Basket (weighted)')
ax3.axhline(1,color='gray',lw=0.8,ls='--')
ax3.set_xlabel('Day'); ax3.set_ylabel('Normalized price')
ax3.set_title('§5.3 Commodity-backed stablecoin\nbasket vs individual assets')
ax3.legend(fontsize=8)

# Bitcoin hash rate (log)
ax4 = axes[1][0]
ax4.semilogy(years_hr, hashrate_EH, 'o-', color='orange', lw=2.5, ms=6)
ax4.fill_between(years_hr, hashrate_EH, alpha=0.3, color='orange')
ax4.set_xlabel('Year'); ax4.set_ylabel('Hash rate (EH/s, log scale)')
ax4.set_title('§5.4 Bitcoin hash rate\n(security = economic deterrence)')
ax4b = ax4.twinx()
ax4b.semilogy(years_hr, np.maximum(attack_cost,0.001), 's--', color='red', lw=1.5,
              ms=5, alpha=0.7, label='51% attack cost $B')
ax4b.set_ylabel('Attack cost ($B, log)', color='red')
ax4b.legend(fontsize=8)

# Triffin dilemma
ax5 = axes[1][1]
gdp_growth = np.linspace(0, 0.05, 100)
# Reserve demand requires running current account deficits
deficit_pct = -0.03 - 1.5*gdp_growth   # more growth -> more global USD demand -> more US deficit
ax5.plot(gdp_growth*100, deficit_pct*100, 'royalblue', lw=2.5)
ax5.axhline(0, color='gray', lw=0.8)
ax5.fill_between(gdp_growth*100, deficit_pct*100, 0, alpha=0.2, color='red', label='Deficit zone')
ax5.set_xlabel('Global GDP growth (%)'); ax5.set_ylabel('US current account (% GDP)')
ax5.set_title("§5 Triffin Dilemma\n(reserve currency = forced deficits)")
ax5.legend(fontsize=9)
ax5.text(2, -5, 'More global growth\n→ more USD demand\n→ deeper US deficit', fontsize=9)

# Power comparison: hash rate vs militaries
ax6 = axes[1][2]
militaries = {
    'USA':     858,
    'China':   225,
    'Russia':  86,
    'UK':      75,
    'India':   72,
    'Germany': 52,
    'BTC 51%\n(2024)': attack_cost[-1],
    'BTC 51%\n(2030est)': attack_cost[-1]*5,
}
cols_mil = ['steelblue','red','darkred','royalblue','orange',
            'green','gold','gold']
names_mil = list(militaries.keys())
vals_mil  = list(militaries.values())
bars = ax6.barh(names_mil, vals_mil, color=cols_mil, alpha=0.85, edgecolor='#333')
ax6.axvline(858, color='blue', ls='--', lw=1, alpha=0.3)
ax6.set_xlabel('Budget/Cost ($B)')
ax6.set_title('§5.4 Bitcoin 51% cost vs\nmilitary budgets ($B, 2024)')
for bar, val in zip(bars, vals_mil):
    ax6.text(val+5, bar.get_y()+bar.get_height()/2,
             f'${val:.0f}B', va='center', fontsize=8)

plt.suptitle('§5: Military-Backed Currency — Petrodollar · Reserve Share · Commodity Crypto · Hash Rate Deterrence',
             fontsize=11,fontweight='bold')
plt.tight_layout(); plt.show()

print('\n§5 Summary: "A currency needs a military (or 580 EH/s)"')
print('  USD: backed by US Navy (Strait of Hormuz) + legal tender law')
print('  BTC: backed by hash rate energy expenditure (~$1.8B/day)')
print('  CommodityCoin: backed by physical collateral + legal redemption')
print('  The weakest link is always: WHO ENFORCES THE CONTRACT?')